# 🔥 LangGraph + Gemini 2.5 Agent

**Goal:** Build your first AI Agent using LangGraph with Gemini!

---

## What is an AI Agent?

```
Chatbot:  User asks \u2192 AI responds
Agent:    User asks \u2192 AI thinks \u2192 uses tools \u2192 acts \u2192 observes \u2192 responds
```

**Agents can USE TOOLS - that's what makes them different!**

In [2]:
# Install dependencies
%pip install langgraph langchain-google-genai -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# 🔥 YOUR FIRST LANGGRAPH AGENT WITH GEMINI
# ============================================================

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
import os

# Load API key from .env file
load_dotenv()
API_KEY = os.environ.get("GEMINI_API_KEY")
print("\u2713 API Key loaded:", bool(API_KEY))

✓ API Key loaded: True


In [4]:
# Create Gemini model - use gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=API_KEY)
print (f"✓✓ Gemini 2.5 Flash model created!")

✓✓ Gemini 2.5 Flash model created!



## Step 1: Create Tools

Tools give your agent "hands" to interact with the world!

In [5]:
# ============================================================
# TOOL 1: Calculator
# ============================================================
@tool
def calculator(expression: str) -> str:
    """Perform math calculations. Input: math expression like 25-8-5"""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# ============================================================
# TOOL 2: Search (using Gemini itself)
# ============================================================
@tool
def search_gemini(query: str) -> str:
    """Use this to search for information. Input: your search query."""
    response = llm.invoke(query)
    return response.content

# Create tools list
tools = [calculator, search_gemini]
print("✓ Tools created: calculator, search_gemini")

✓ Tools created: calculator, search_gemini


## Step 2: Create the Agent

Using `create_react_agent` - this is the ReAct pattern (Reason + Act)

In [6]:
# Create LangGraph ReAct agent
agent = create_react_agent(llm, tools)
print(f"✓ LangGraph ReAct Agent created!")


✓ LangGraph ReAct Agent created!


/var/folders/2l/8f5zhycd0hbb538hb0pht_5w0000gn/T/ipykernel_10463/9702717.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


## Step 3: Test the Agent!

In [7]:
# ============================================================
# TEST 1: Math Problem (Agent uses Calculator tool)
# ============================================================
print("="*50)
print("TEST 1: Math Problem")
print("="*50)

result = agent.invoke({
    "messages": [("human", "If I have 25 apples, give 8 to John, 5 to Mary. How many apples are left?")]
})

print("\n💬 Question: If I have 25 apples, give 8 to John, 5 to Mary. How many apples are left?")
print(f"🤔 Answer: {result['messages'][-1].content}")




TEST 1: Math Problem

💬 Question: If I have 25 apples, give 8 to John, 5 to Mary. How many apples are left?
🤔 Answer: You will have 12 apples left.


In [26]:
result = agent.invoke({"messages": [("user","india capital?")]})
print(f"Answer: {result['messages'][-1].content [0]['text']}")

Answer: New Delhi


In [8]:
# ============================================================
# TEST 2: Information Search (Agent uses Search tool)
# ============================================================
print("\n" + "="*50)
print("TEST 2: Information Search")
print("="*50)

result2 = agent.invoke({
    "messages": [("human", "What is RAG in machine learning?")]
})

print("\n💬 Question: What is RAG in machine learning?")
print(f" Answer: {result2['messages'][-1].content[:300]}...")
print(f"Answer: {result2['messages'][-1].content[0]['text']}...")



TEST 2: Information Search

💬 Question: What is RAG in machine learning?
 Answer: [{'type': 'text', 'text': 'RAG stands for Retrieval Augmented Generation. It\'s a machine learning technique, primarily used with Large Language Models (LLMs), to enhance the quality, factual accuracy, and relevance of generated text.\n\nHere\'s a breakdown of why it\'s used and how it works:\n\n**Why RAG is Needed:**\nLLMs, despite their vast training data, can sometimes "hallucinate" (make up facts), provide out-of-date information, lack domain-specific knowledge, and cannot cite sources. RAG helps overcome these limitations.\n\n**How RAG Works (Two Main Phases):**\n\n1.  **Retrieval:**\n    *   An external knowledge base (e.g., documents, web pages, databases) is indexed. This involves breaking down documents into smaller chunks and converting them into numerical representations called **embeddings**, which are stored in a **vector database**.\n    *   When a user asks a question, the query is also co

In [ ]:
# ============================================================
# TEST 3: Multi-step problem
# ============================================================
print("\n" + "="*50)
print("TEST 3: Multi-step Problem")
print("="*50)

result3 = agent.invoke({
    "messages": [("human", "If a train travels 120 km in 2 hours, and then 180 km in 3 hours, what is its average speed?")]
})

print("\n💬 Question: Train problem")
#print(f"Answer: {result3['messages'][-1].content}")

print(f"Answer: {result3['messages'][-1].content[0]['text']}")


TEST 3: Multi-step Problem

💬 Question: Train problem
Answer: The train's average speed is 60 km/h.


---

## 🎯 ADVANCED: Custom LangGraph Workflow

In [10]:
# ============================================================
# CUSTOM STATEGRAPH - Build your own agent workflow
# ============================================================

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Define your custom state schema
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    question: str
    answer: str
    steps: list

# Define nodes (functions that process state)
def analyze(state: AgentState) -> AgentState:
    """Analyze the question"""
    question = state["question"]
    print(f"🔍 Analyzing: {question}")
    return {"steps": ["analyzed"]}

def generate(state: AgentState) -> AgentState:
    """Generate answer using Gemini"""
    response = llm.invoke(state["question"])
    answer = response.content
    print(f"💡 Generated answer")
    return {"answer": answer, "steps": ["generated"]}

def improve(state: AgentState) -> AgentState:
    """Improve the answer"""
    answer = state["answer"]
    improved = f"Here is a detailed answer:\n\n{answer}"
    print(f"✅ Answer improved")
    return {"answer": improved, "steps": ["improved"]}

# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("analyze", analyze)
workflow.add_node("generate", generate)
workflow.add_node("improve", improve)

# Define edges (the flow)
workflow.set_entry_point("analyze")
workflow.add_edge("analyze", "generate")
workflow.add_edge("generate", "improve")
workflow.add_edge("improve", END)

# Compile
custom_agent = workflow.compile()

# Run
result = custom_agent.invoke({
    "question": "What is a neural network in simple terms?",
    "messages": []
})

print("\n" + "="*50)
print("CUSTOM AGENT RESULT:")
print("="*50)
print(result["answer"])

🔍 Analyzing: What is a neural network in simple terms?
💡 Generated answer
✅ Answer improved

CUSTOM AGENT RESULT:
Here is a detailed answer:

Imagine a **neural network** like a simplified, digital "brain" that learns from examples, much like a child learns to identify objects.

Here's the breakdown in simple terms:

1.  **Inspired by the Brain:** It's loosely modeled after the human brain, which is made of billions of interconnected neurons. A neural network has "artificial neurons" (also called **nodes**) connected in layers.

2.  **Layers of Processing:**
    *   **Input Layer:** This is where you feed in your data (e.g., the pixels of an image, the words in a sentence, numbers).
    *   **Hidden Layers:** These are the "thinking" layers in between. Each node in these layers takes inputs from the previous layer, does a simple calculation, and passes its result to the next layer. This is where the magic happens – the network figures out complex patterns.
    *   **Output Layer:** Thi

---

## 📊 Interview Punch

> **"I built an AI agent using LangGraph with Gemini 2.5. It's a ReAct agent that uses tools (calculator + search) to answer questions. The agent decides when to use tools by reasoning through the problem, acting on them, and observing results - this is the ReAct pattern. LangGraph lets me build complex agent workflows with custom state management - this is how production AI agents are built at companies like Databricks with their KARL agent."**

---

## 📝 Key Concepts Summary

| Concept | What It Does |
|---------|---------------|
| **LangGraph** | Build agent workflows as graphs |
| **create_react_agent** | Pre-built ReAct agent |
| **@tool decorator** | Create tools for agents |
| **StateGraph** | Custom agent with state management |
| **ReAct** | Reason + Act + Observe loop |

---

## ✅ Checkpoint

- [ ] Built LangGraph agent with Gemini
- [ ] Used @tool decorator to create tools
- [ ] Tested agent with math problem
- [ ] Tested agent with search
- [ ] Built custom StateGraph workflow
- [ ] Can explain ReAct loop to interviewer

---

*🔥 You just built a real AI Agent! This is the foundation for GCP AI Prep.*